# Demo: spatial join across estados / municipios / focos_de_fogo

Draw an area of interest on the map, then query `focos_de_fogo` joined to `municipios` and `estados` through their foreign keys, filtered to that area.

In [1]:
import os

import geopandas as gpd
import leafmap
import pandas as pd
from dotenv import load_dotenv
from shapely.geometry import shape
from sqlalchemy import create_engine, text


In [2]:
load_dotenv()

engine = create_engine(
    "mysql+pymysql://{user}:{pswd}@{host}:{port}/{db}".format(
        user=os.getenv("MARIADB_USER"),
        pswd=os.getenv("MARIADB_PASSWORD"),
        host=os.getenv("HOST"),
        port=os.getenv("PORT"),
        db=os.getenv("MARIADB_DATABASE"),
    )
)


## 1. Select an area

Use the rectangle or polygon tool on the left-hand toolbar to draw an area of interest on the map below.

In [3]:
m = leafmap.Map(center=[-14, -52], zoom=4)
m.add_basemap("OpenStreetMap")
m


OpenStreetMap has been already added before.


Map(center=[-14, -52], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## 2. Query fire focuses within the drawn area

This runs a single SQL query that joins `focos_de_fogo` → `municipios` → `estados` through their foreign keys (`municipio_id`, `estado_id`), filtered with `ST_Intersects` against the drawn shape's geometry.

In [4]:
def query_focos_in_roi(m: leafmap.Map) -> gpd.GeoDataFrame:
    if m.user_roi is None:
        raise ValueError("Draw an area on the map above before running this query.")

    roi_wkt = shape(m.user_roi["geometry"]).wkt

    query = text(
        """
        SELECT
            f.id_foco_bdq,
            f.satelite,
            f.data_hora_gmt,
            f.frp,
            f.risco_fogo,
            f.bioma,
            m.nome AS municipio,
            e.nome AS estado,
            e.regiao,
            ST_AsBinary(f.geometry) AS geometry
        FROM focos_de_fogo f
        JOIN municipios m ON m.id = f.municipio_id
        JOIN estados e ON e.id = m.estado_id
        WHERE ST_Intersects(f.geometry, ST_GeomFromText(:roi_wkt, 4674))
        """
    )

    with engine.connect() as conn:
        df = pd.read_sql(query, con=conn, params={"roi_wkt": roi_wkt})

    return gpd.GeoDataFrame(
        df.drop(columns=["geometry"]),
        geometry=gpd.GeoSeries.from_wkb(df["geometry"]),
        crs="EPSG:4674",
    )


focos_in_roi = query_focos_in_roi(m)
print(f"{len(focos_in_roi)} focos de fogo found in the selected area")
focos_in_roi.drop(columns="geometry")


117 focos de fogo found in the selected area


,id_foco_bdq,satelite,data_hora_gmt,frp,risco_fogo,bioma,municipio,estado,regiao
0,1829998285,NPP-375,2026-08-02 17:09:00,2.6,0.45,Mata Atlântica,UBATUBA,SÃO PAULO,SUDESTE
1,1830002730,NOAA-20,2026-08-02 17:30:00,3.0,1.00,Mata Atlântica,LORENA,SÃO PAULO,SUDESTE
2,1829999112,NPP-375,2026-08-02 17:09:00,2.6,1.00,Mata Atlântica,LORENA,SÃO PAULO,SUDESTE
3,1830001183,NOAA-21,2026-08-02 16:35:00,2.0,1.00,Mata Atlântica,GUARATINGUETÁ,SÃO PAULO,SUDESTE
4,1830002056,AQUA_M-T,2026-08-02 18:38:00,5.4,1.00,Mata Atlântica,CAMPANHA,MINAS GERAIS,SUDESTE
...,...,...,...,...,...,...,...,...,...
112,1829823200,NOAA-21,2026-08-02 04:06:00,0.8,1.00,Mata Atlântica,ITU,SÃO PAULO,SUDESTE
113,1830000659,NOAA-21,2026-08-02 16:35:00,2.3,1.00,Mata Atlântica,NOVA ODESSA,SÃO PAULO,SUDESTE
114,1830000949,NOAA-21,2026-08-02 16:35:00,3.4,1.00,Mata Atlântica,SALTO,SÃO PAULO,SUDESTE
115,1830058694,NOAA-21,2026-08-03 03:49:00,2.4,NaN,Mata Atlântica,LEME,SÃO PAULO,SUDESTE


## 3. Visualize the joined results

Click a point to see its joined `municipio`/`estado` attributes, pulled in purely through the foreign-key relationships.

In [6]:
m.add_gdf(
    focos_in_roi.to_crs("EPSG:4326"),
    layer_name="Focos de fogo",
    info_mode="on_click",
)
m


Map(bottom=37303.0, center=[-22.649502094242195, -44.84619140625001], controls=(ZoomControl(options=['position…